<!-- # Spatial Disorder & Localisation

Real-space analysis of lower polariton localisation in a 2-D
Gaussian-correlated disorder potential.

**Contents**
1. Single realisation — disorder landscape and lowest eigenmodes
2. Sigma sweep — effective interaction strength vs disorder amplitude
3. (Optional) Xi sweep — correlation-length dependence of IPR -->


In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from polaritons.real_space import (
	gaussian_correlated_disorder,
	lp_hamiltonian,
	solve_low_modes,
	normalize_mode,
	ipr,
	mode_area,
	g_eff_lowest_mode,
)
from polaritons.io import save_result
from polaritons.parameters import DEFAULT_PARAMS

# ---------------------------------------------------------------------------
# Matplotlib style
# ---------------------------------------------------------------------------
CMAP_NAME = "magma"
CMAP = plt.get_cmap(CMAP_NAME)

plt.rcParams.update({
	'font.family': 'serif',
	'font.size': 13,
	'axes.labelsize': 14,
	'axes.titlesize': 14,
	'figure.titlesize': 16,
	'image.cmap': CMAP_NAME,
})

REAL_SPACE_PLOTS_DIR = Path("Plots") / "Real_Space"
REAL_SPACE_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Physical constants (SI, eV-based)
# ---------------------------------------------------------------------------
hbar = 6.582119569e-16   # eV·s
m0   = 5.685630e-12      # eV·s²/m²  (free electron rest mass)

# ---------------------------------------------------------------------------
# System parameters
# ---------------------------------------------------------------------------
N = 256                  # grid points per side
L = 40e-6                # system size (m)

M_lp = 1e-5 * m0         # LP effective mass
xi   = 20e-9             # disorder correlation length (m)

# Express everything as a *relative* interaction strength: g_eff / g_lp.
# With g_lp set to 1, g_eff equals the IPR (= 1/mode_area), which is the
# dimensionless suppression/enhancement of the lowest-mode interaction
# relative to the homogeneous (clean) case.
g_lp = 1.0
G_PLOT_LABEL = r"relative interaction strength  $g_{\rm eff}/g_{\rm lp}$"
AREA_PLOT_LABEL = r"mode area $A_{\rm eff}$ (µm$^2$)"

print(f"Grid: {N}×{N},  L={L*1e6:.0f} µm,  dx={L/N*1e9:.1f} nm")
print(f"M_lp = {M_lp:.3e} eV·s²/m²   xi = {xi*1e9:.0f} nm")
print(f"g_lp = {g_lp}  (relative units)")


## 1 — Single realisation

Visualise the disorder landscape and the three lowest polariton eigenmodes.


In [ ]:
SEED          = 1
N_MODES       = 6
SIGMA_VALUES  = np.array([0.2, 0.5, 1.0, 2.0, 5.0, 10.0]) * 1e-3   # eV
dx            = L / N

x_um = np.linspace(0, L * 1e6, N)
X, Y = np.meshgrid(x_um, x_um, indexing="ij")


def _sigma_slug(sigma_eV: float) -> str:
	# 0.5 meV -> "0p5meV"
	return f"{sigma_eV * 1e3:g}meV".replace(".", "p")


# Pre-compute disorder + modes for every sigma so we can build a composite
# figure with consistent per-sigma color scales.
landscapes  = []   # list of (sigma, V) pairs
mode_stacks = []   # list of (sigma, energies, modes) triples

print(f"  sigma (meV)   mode    E (meV)        A_eff (µm²)     g_eff / g_lp")
for sigma in SIGMA_VALUES:
	V = gaussian_correlated_disorder(N, L, float(sigma), xi, seed=SEED)
	H = lp_hamiltonian(N, L, M_lp, V, hbar)
	E, psi = solve_low_modes(H, n_modes=N_MODES)
	modes = [normalize_mode(psi[:, n].reshape(N, N), dx) for n in range(psi.shape[1])]

	landscapes.append((float(sigma), V))
	mode_stacks.append((float(sigma), E, modes))

	for n, m in enumerate(modes):
		A_eff_um2 = mode_area(m, dx) * 1e12
		rel_g     = g_lp * ipr(m, dx) / ipr(modes[0], dx)  # rel. to ground; g_lp=1
		print(f"  {sigma*1e3:9.2f}    {n}    {1e3*E[n]:10.4f}    {A_eff_um2:12.3f}    {rel_g:12.4e}")

# ---------------------------------------------------------------------------
# Per-sigma row figures: V + |psi|^2 for the N_MODES lowest modes,
# probability densities sharing the same color scale within each row.
# ---------------------------------------------------------------------------
panel_w, panel_h = 3.0, 2.8
ncols = 1 + N_MODES

per_sigma_paths = []
for (sigma, V), (_, E, modes) in zip(landscapes, mode_stacks):
	fig, axes = plt.subplots(1, ncols, figsize=(panel_w * ncols, panel_h),
							 constrained_layout=True)

	# Disorder landscape (its own color scale).
	im_V = axes[0].pcolormesh(X, Y, 1e3 * V, shading="auto", cmap=CMAP_NAME)
	axes[0].set_title(f"V  σ={sigma*1e3:.2g} meV")
	axes[0].set_xlabel("x (µm)")
	axes[0].set_ylabel("y (µm)")
	plt.colorbar(im_V, ax=axes[0], fraction=0.046, label="V (meV)")

	# Probability densities: shared row-level color scale.
	prob_densities = [np.abs(m) ** 2 for m in modes]
	vmin = 0.0
	vmax = max(p.max() for p in prob_densities)

	for n, prob in enumerate(prob_densities):
		ax = axes[1 + n]
		im = ax.pcolormesh(X, Y, prob, shading="auto",
						   cmap=CMAP_NAME, vmin=vmin, vmax=vmax)
		ax.set_title(f"mode {n}\nE={1e3*E[n]:.3f} meV")
		ax.set_xlabel("x (µm)")
		ax.set_yticklabels([])
	# One shared colorbar for the row of probability densities.
	plt.colorbar(im, ax=axes[1:], fraction=0.025, pad=0.01,
				 label=r"$|\psi|^2$ (m$^{-2}$)")

	out_path = REAL_SPACE_PLOTS_DIR / f"modes_sigma_{_sigma_slug(sigma)}.png"
	fig.savefig(out_path, dpi=150, bbox_inches="tight")
	per_sigma_paths.append(out_path)
	print(f"Saved  {out_path}")
	plt.show()
	plt.close(fig)

# ---------------------------------------------------------------------------
# Composite figure: one row per sigma, columns = V + N_MODES eigenmodes.
# Probability densities share one global color scale so disorder-driven
# localisation is directly comparable across rows.
# ---------------------------------------------------------------------------
nrows = len(SIGMA_VALUES)
fig, axes = plt.subplots(nrows, ncols,
						 figsize=(panel_w * ncols, panel_h * nrows),
						 constrained_layout=True)
if nrows == 1:
	axes = axes[np.newaxis, :]

global_vmax = max(np.abs(m) ** 2 for (_, _, modes) in mode_stacks for m in modes).max()

for row, ((sigma, V), (_, E, modes)) in enumerate(zip(landscapes, mode_stacks)):
	axes[row, 0].pcolormesh(X, Y, 1e3 * V, shading="auto", cmap=CMAP_NAME)
	axes[row, 0].set_ylabel(f"σ={sigma*1e3:.2g} meV\ny (µm)")
	if row == 0:
		axes[row, 0].set_title("disorder V")
	if row == nrows - 1:
		axes[row, 0].set_xlabel("x (µm)")

	for n, m in enumerate(modes):
		ax = axes[row, 1 + n]
		im = ax.pcolormesh(X, Y, np.abs(m) ** 2, shading="auto",
						   cmap=CMAP_NAME, vmin=0.0, vmax=global_vmax)
		if row == 0:
			ax.set_title(f"mode {n}")
		if row == nrows - 1:
			ax.set_xlabel("x (µm)")
		ax.set_yticklabels([])

fig.colorbar(im, ax=axes[:, 1:], fraction=0.012, pad=0.01,
			 label=r"$|\psi|^2$ (m$^{-2}$)")
composite_path = REAL_SPACE_PLOTS_DIR / "mode_evolution.png"
fig.savefig(composite_path, dpi=150, bbox_inches="tight")
print(f"Saved  {composite_path}")
plt.show()
plt.close(fig)


## 2 — Sigma sweep

Average `g_eff` of the ground mode over multiple disorder realisations
as a function of disorder amplitude σ.  Results are saved to `Results/`.


In [ ]:
from polaritons.io import save_result, make_sweep_stem
from polaritons.parameters import DEFAULT_PARAMS

# ---------------------------------------------------------------------------
# Sweep configuration
# ---------------------------------------------------------------------------
sigmas = np.array([0.0, 0.02e-3, 0.05e-3, 0.1e-3, 0.2e-3, 0.5e-3, 1.0e-3])  # eV
n_realisations = 20

# ---------------------------------------------------------------------------
# Compute: relative interaction strength = g_eff / g_lp = IPR (since g_lp=1).
# Reported as the lowest-mode value.
# ---------------------------------------------------------------------------
g_mean = np.zeros(len(sigmas))
g_std  = np.zeros(len(sigmas))

for si, s in enumerate(sigmas):
	vals = [
		g_eff_lowest_mode(N, L, M_lp, s, xi, hbar, g_lp=g_lp, seed=seed)
		for seed in range(n_realisations)
	]
	g_mean[si] = np.mean(vals)
	g_std[si]  = np.std(vals)
	print(f"  σ={s*1e3:5.2f} meV  →  g_eff/g_lp = {g_mean[si]:.4e} ± {g_std[si]:.4e}")

# ---------------------------------------------------------------------------
# Save results
# ---------------------------------------------------------------------------
result_array = np.vstack([sigmas, g_mean, g_std])   # shape (3, len(sigmas))

extra_meta = {
	"rows"           : ["sigma_eV", "g_rel_mean", "g_rel_std"],
	"N"              : N,
	"L_m"            : L,
	"M_lp_eVs2m2"    : M_lp,
	"xi_m"           : xi,
	"g_lp"           : g_lp,
	"plot_units"     : "g_eff / g_lp (dimensionless)",
	"n_realisations" : n_realisations,
	"hbar_eVs"       : hbar,
}
stem = make_sweep_stem("geff_sigma", DEFAULT_PARAMS, extra={"xi": xi, "N": N})
save_result(result_array, "Results/interaction_strengths", stem, DEFAULT_PARAMS, extra_meta)

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(1e3 * sigmas, g_mean, yerr=g_std,
			marker='o', capsize=4, linewidth=1.5,
			color=CMAP(0.72), markerfacecolor=CMAP(0.88))
ax.set_xlabel(r"$\sigma$ (meV)")
ax.set_ylabel(G_PLOT_LABEL)
ax.set_title(r"Lowest-mode relative interaction vs disorder amplitude")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


## 3 — Xi sweep  *(optional)*

Repeat the ground-mode IPR analysis across a range of correlation lengths ξ
at fixed disorder amplitude.


In [ ]:
xis            = np.array([5e-9, 10e-9, 20e-9, 40e-9, 80e-9])  # m
sigma_fixed    = 0.2e-3   # eV — held constant
n_realisations = 10

xi_g_mean = np.zeros(len(xis))
xi_g_std  = np.zeros(len(xis))

for xi_i, xi_val in enumerate(xis):
	vals = [
		g_eff_lowest_mode(N, L, M_lp, sigma_fixed, xi_val, hbar, g_lp=g_lp, seed=seed)
		for seed in range(n_realisations)
	]
	xi_g_mean[xi_i] = np.mean(vals)
	xi_g_std[xi_i]  = np.std(vals)
	print(f"  ξ={xi_val*1e9:5.0f} nm  →  g_eff/g_lp = {xi_g_mean[xi_i]:.4e} ± {xi_g_std[xi_i]:.4e}")

# Save
result_xi = np.vstack([xis, xi_g_mean, xi_g_std])
extra_xi = {
	"rows"           : ["xi_m", "g_rel_mean", "g_rel_std"],
	"sigma_eV"       : sigma_fixed,
	"N"              : N,
	"L_m"            : L,
	"M_lp_eVs2m2"    : M_lp,
	"g_lp"           : g_lp,
	"plot_units"     : "g_eff / g_lp (dimensionless)",
	"n_realisations" : n_realisations,
}
stem_xi = make_sweep_stem("geff_xi", DEFAULT_PARAMS, extra={"sigma": sigma_fixed, "N": N})
save_result(result_xi, "Results/interaction_strengths", stem_xi, DEFAULT_PARAMS, extra_xi)

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(xis * 1e9, xi_g_mean, yerr=xi_g_std,
			marker='s', capsize=4, linewidth=1.5,
			color=CMAP(0.65), markerfacecolor=CMAP(0.9))
ax.set_xlabel(r"$\xi$ (nm)")
ax.set_ylabel(G_PLOT_LABEL)
ax.set_title(fr"Lowest-mode relative interaction vs correlation length  ($\sigma$={sigma_fixed*1e3:.1f} meV)")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
